<center><h1>Satish_Manvika_HW7</h1></center>

Name: Manvika Satish
<br>
Github Username: manvsatish
<br>
USC ID: 5582061880

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, hamming_loss, silhouette_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 552
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
print("Packages imported successfully.")

Packages imported successfully.


### (a) Download the Anuran Calls (MFCCs) Data Set

In [4]:
possible_paths = [Path("../data/Frogs_MFCCs.csv"), Path("data/Frogs_MFCCs.csv"),] #locate & load dataset
data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("Frogs_MFCCs.csv was not found in the data folder.")

frogs = pd.read_csv(data_path)
print(f"Dataset path: {data_path}")
print(f"Dataset shape: {frogs.shape}")
print(f"Number of missing values: {frogs.isna().sum().sum()}")
display(frogs.head())
print("\nColumn names:")
print(frogs.columns.tolist())

feature_columns = [column for column in frogs.columns if column.startswith("MFCCs_")] #separate 22 predictors, 3 labels, and id
label_columns = ["Family", "Genus", "Species"]
X = frogs[feature_columns]
y = frogs[label_columns]

label_combinations = y.astype(str).agg("_".join, axis=1) # combined label triplet to keep class distributions

#random selecttion 70% for training and 30% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=label_combinations)
print(f"Number of predictors: {X.shape[1]}")
print(f"Label columns: {label_columns}")
print(f"X training shape: {X_train.shape}")
print(f"X testing shape: {X_test.shape}")
print(f"y training shape: {y_train.shape}")
print(f"y testing shape: {y_test.shape}")
print(f"Training percentage: {100 * len(X_train) / len(X):.1f}%")
print(f"Testing percentage: {100 * len(X_test) / len(X):.1f}%")
print(f"Random seed: {RANDOM_STATE}")

Dataset path: ../data/Frogs_MFCCs.csv
Dataset shape: (7195, 26)
Number of missing values: 0


,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,MFCCs_11,MFCCs_12,MFCCs_13,MFCCs_14,MFCCs_15,MFCCs_16,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,0.188654,-0.075622,-0.156436,0.082245,0.135752,-0.024017,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,0.270958,-0.095004,-0.254341,0.022786,0.163320,0.012022,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,0.266064,-0.072827,-0.237384,0.050791,0.207338,0.083536,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,0.267279,-0.162258,-0.317084,-0.011567,0.100413,-0.050224,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,Leptodactylidae,Adenomera,AdenomeraAndre,1
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,0.332695,-0.100749,-0.298524,0.037439,0.219153,0.062837,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,Leptodactylidae,Adenomera,AdenomeraAndre,1



Column names:
['MFCCs_ 1', 'MFCCs_ 2', 'MFCCs_ 3', 'MFCCs_ 4', 'MFCCs_ 5', 'MFCCs_ 6', 'MFCCs_ 7', 'MFCCs_ 8', 'MFCCs_ 9', 'MFCCs_10', 'MFCCs_11', 'MFCCs_12', 'MFCCs_13', 'MFCCs_14', 'MFCCs_15', 'MFCCs_16', 'MFCCs_17', 'MFCCs_18', 'MFCCs_19', 'MFCCs_20', 'MFCCs_21', 'MFCCs_22', 'Family', 'Genus', 'Species', 'RecordID']
Number of predictors: 22
Label columns: ['Family', 'Genus', 'Species']
X training shape: (5036, 22)
X testing shape: (2159, 22)
y training shape: (5036, 3)
y testing shape: (2159, 3)
Training percentage: 70.0%
Testing percentage: 30.0%
Random seed: 552


### (b) Train a classifier for each label

#### (i) Research

#### (ii) Train a SVM for each of the labels

#### (iii) Repeat 1(b)ii with L1-penalized SVMs

#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

### (b) Determine which family is the majority

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

## 3. ISLR 12.6.2